In [1]:
import os
import xarray as xr
import numpy as np

In [2]:
# === Filename builders ===
def build_arise_path(ens_num, VAR, date_range):
    base = (
        "/glade/campaign/cesm/collections/ARISE-SAI-1.5/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens_num}.cam.h4.{VAR}.{date_range}.nc"
    )
    return base


def build_ssp245_path(ens_num, VAR, date_range):
    base = (
        "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}/atm/proc/tseries/hour_1/"
        f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens_num}.cam.h4.{VAR}.{date_range}.nc"
    )
    return base

In [3]:
# === File list generator ===
def get_file_list(scenario, ens_num, VAR):
    files = []
    num = f"{ens_num:02d}"

    if scenario == "ARISE":
        if ens_num in [5, 8, 9]:  # Yearly
            for year in range(2035, 2070 if ens_num in [8, 9] else 2069):
                start = f"{year}010100"
                end = f"{year+1}010100"
                date_range = f"{start}-{end}"
                files.append(build_arise_path(num, VAR, date_range))
            if ens_num in [5]:
                files.append(build_arise_path(num, VAR, "2069010100-2069123100"))
        else:  # Decadal
            for start, end in [(2035, 2045), (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_arise_path(num, VAR, date_range))

    elif scenario == "SSP245":
        if ens_num <= 5:  # through 2100
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2075)]:
                date_range = f"{start}010100-{end}010100"
                files.append(build_ssp245_path(num, VAR, date_range))
        else:  # ends in 20691231
            for start, end in [(2015, 2025), (2025, 2035), (2035, 2045), (2045, 2055), (2055, 2065), (2065, 2069)]:
                date_range = (
                    f"{start}010100-{end}123100"
                    if start == 2065
                    else f"{start}010100-{end}010100"
                )
                files.append(build_ssp245_path(num, VAR, date_range))

    return files

In [4]:
def calculate_monthly_mean_8hrdailymax_NO2(start_date, end_date, o3_surf, no2_surf):
    daterange = xr.date_range(start_date, end_date, calendar="noleap", use_cftime=True)

    MDA8_NO2 = xr.DataArray(
        np.nan,
        dims=["time", "lat", "lon"],
        coords={"time": daterange, "lat": no2_surf.lat, "lon": no2_surf.lon},
    )

    for i, date in enumerate(daterange):
        date = daterange[i].strftime("%Y-%m-%d")

        o3_day = o3_surf.sel(time=slice(date + " 00:00:00", date + " 23:00:00"))
        no2_day = no2_surf.sel(time=slice(date + " 00:00:00", date + " 23:00:00"))

        if o3_day.time.size < 24:
            continue

        # Rolling 8h mean for O3
        o3_rolling = o3_day.rolling(time=8, center=False).mean()

        # Max 8h O3 and index
        idx_of_max = o3_rolling.argmax(dim="time")  # (lat, lon)

        # Now build indices for 8-hour window
        idx_window = xr.DataArray(
            np.arange(-7, 1),
            dims=["window"],
        ) + idx_of_max.expand_dims(window=8)

        # Mask invalid indices
        valid = (idx_window >= 0) & (idx_window < o3_day.time.size)
        idx_window = idx_window.where(valid, 0)

        # Use isel smartly
        no2_selected = no2_day.isel(time=idx_window)  # shape (window, lat, lon)

        # Now average over the 8-hour window
        no2_8hr_mean = no2_selected.where(valid).mean(dim="window", skipna=True)

        MDA8_NO2[i, :, :] = no2_8hr_mean

    monthly_mean_no2 = MDA8_NO2.resample(time="ME").mean()

    return monthly_mean_no2


In [5]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/NO2_MDA8/"
#SCENARIOS = ["ARISE", "SSP245"]
SCENARIOS = ["ARISE"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(9, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        file_list_o3 = get_file_list(scenario, ens_num, "O3_SRF")
        file_list_no2 = get_file_list(scenario, ens_num, "NO2_SRF")
        monthly_means = []

        for f_no2, f_o3 in zip(file_list_no2, file_list_o3):
            for f in [f_no2, f_o3]:
                if not os.path.exists(f):
                    raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f_no2)}")
            ds_no2 = xr.open_dataset(f_no2)["NO2_SRF"]
            ds_o3 = xr.open_dataset(f_o3)["O3_SRF"]

            start_date = str(ds_o3.time[0].values)[:10]
            end_date = str(ds_o3.time[-1].values)[:10]
            end_time = str(ds_o3.time[-1].values)[11:16]
            midnight = "00:00"

            if end_time == midnight:
                print("changing final time step")
                end_date = str(ds_o3.time[-2].values)[:10]

            mm = calculate_monthly_mean_8hrdailymax_NO2(start_date, end_date, ds_o3, ds_no2)

            if scenario == "SSP245":
                # Trim to 2069-12-31
                mm = mm.sel(time=slice("2020-01-01", "2069-12-31"))

            if scenario == "ARISE" and ens_num in [8, 9]:
                # Trim to 2069-12-31
                mm = mm.sel(time=slice("2035-01-01", "2069-12-31"))

            monthly_means.append(mm)

        if monthly_means:
            combined = xr.concat(monthly_means, dim="time")

            if scenario == "ARISE":
                dates = "20350101-20691231"
            elif scenario == "SSP245":
                dates = "20200101-20691231"

            out_file = f"MDA8_NO2_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")


Processing ARISE, Ensemble 09
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2035010100-2036010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2036010100-2037010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2037010100-2038010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2038010100-2039010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2039010100-2040010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2040010100-2041010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2041010100-2042010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2042010100-2043010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2043010100-2044010100.nc
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.009.cam.h4.NO2_SRF.2044010100-2045010100.nc
Reading b.